In [23]:
import math
import numpy as nm
import matplotlib.pyplot as plt
import random

In [ ]:
class Value:
    
    def __init__(self, data, _children = ()):
        self.data = data
        self.grad = 0.0
        self._children = set(_children)
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other,))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other, ))
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float power"
        out = Value(self.data ** other, (self, ))
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        out = Value(math.tanh(self.data), (self, ))
        def _backward():
            self.grad += (1.0 - out.data ** 2) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = Value(0.0 if self.data < 0.0 else self.data, (self, ))
        def _backward():
            self.grad += (out.data > 0.0) * out.grad
        out._backward = _backward
        return out
    
    def sigmoid(self):
        out = Value(1.0 / (1.0 + math.exp(-self.data)), (self, ))
        def _backward():
            self.grad += out.data * (1.0 - out.data) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        used = set()
        order = []

        def topsort(v):
            used.add(v)
            for u in v._children:
                if u not in used:
                    topsort(u)
            order.append(v)

        topsort(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

    def __truediv__(self, other):
        return self * (other ** -1)
    
    def __rtruediv__(self, other):
        return (self ** -1) * other

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return (-self) + other

    def __repr__(self):
        return f'Value(data = {self.data})'

In [145]:
class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):
        return []
    
class Neuron(Module):

    def __init__(self, nin, act = None):
        self._nin = nin
        self.ws = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.bias = Value(random.uniform(-1, 1))
        self._act = act
    
    def __call__(self, x):
        val = sum((wi * xi for wi, xi in zip(self.ws, x)), self.bias)
        if self._act is None:
            return val
        if self._act == 'tanh':
            return val.tanh()
        if self._act == 'relu':
            return val.relu()
        if self._act == 'sigmoid':
            return val.sigmoid()
    
    def parameters(self):
        return self.ws + [self.bias]
    
    def __repr__(self):
        return f'Neuron(nin = {self._nin}, act = {self._act})'

class Layer(Module):

    def __init__(self, nin, nout, act = None):
        self._nin = nin
        self._nout = nout
        self._act = act
        self.neurons = [Neuron(nin, act) for _ in range(nout)]
    
    def __call__(self, x):
        assert len(x) == self._nin, f'size of x must be {self._nin}'
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]
    
    def __repr__(self):
        return f'Layer(nin = {self._nin}, nout = {self._nout}, act = {self._act})'

class MLP(Module):

    def __init__(self, nin, nouts, out_acts = None):
        sizes = [nin] + list(nouts)
        self._nin = nin
        self.layers = [Layer(sizes[i], sizes[i+1], None if out_acts is None else out_acts[i]) for i in range(len(sizes)-1)]
    
    def __call__(self, x):
        assert len(x) == self._nin, f'size of x must be {self._nin}'
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    
    def __repr__(self):
        return f'MLP(\n   ' +f',\n   '.join(str(layer) for layer in self.layers)+f'\n)'